# Introduction to Natural Language Processing (NLP)

**Python Machine Learning Course — Week 25**
**Learn and Help | Academic Year 2025–2026**

This notebook walks you through the NLP pipeline step by step:
1. Tokenization — breaking text into words
2. Stopword removal — filtering out filler words
3. Feature extraction — Bag of Words and TF-IDF
4. Sentiment analysis with **scikit-learn** (classic ML)
5. Sentiment analysis with **Keras** (neural network)
6. Comparing both approaches

---

## Part 1: Tokenization — Breaking Text into Pieces

Tokenization is the first step in any NLP pipeline. We split text into individual words (tokens) so we can analyze them.

In [ ]:
# =============================================
# TOKENIZATION — See how text gets split into words
# =============================================

from sklearn.feature_extraction.text import CountVectorizer

# Try changing this sentence to anything you want!
my_text = ["The cat sat on the mat and the cat liked the mat"]

vectorizer = CountVectorizer()
vectorizer.fit(my_text)

print("Original text:", my_text[0])
print()
print("Tokens (unique words):", list(vectorizer.get_feature_names_out()))
print()

# See the word counts
word_counts = vectorizer.transform(my_text).toarray()[0]
print("Word counts:")
for word, count in zip(vectorizer.get_feature_names_out(), word_counts):
    print(f"  '{word}' appears {count} time(s)")

### Try It Yourself!

Change the text in the cell above to:
- A sentence from your favorite book or movie
- A text message you might send to a friend
- A long paragraph from a news article

Notice how the tokenizer handles different inputs.

## Part 2: Stopword Removal — Filtering Out the Filler

Stopwords are common words like "the", "is", "and" that appear everywhere but don't help with classification.

In [ ]:
# =============================================
# STOPWORD REMOVAL — See what gets filtered out
# =============================================

from sklearn.feature_extraction.text import CountVectorizer

text = ["The movie was really a great and fun experience for the whole family"]

# Without stopword removal
vec_all = CountVectorizer()
vec_all.fit(text)
print("ALL words:", list(vec_all.get_feature_names_out()))
print(f"  Total: {len(vec_all.get_feature_names_out())} words")

print()

# With stopword removal
vec_clean = CountVectorizer(stop_words='english')
vec_clean.fit(text)
print("AFTER removing stopwords:", list(vec_clean.get_feature_names_out()))
print(f"  Total: {len(vec_clean.get_feature_names_out())} words")

print()

# What was removed?
all_words = set(vec_all.get_feature_names_out())
clean_words = set(vec_clean.get_feature_names_out())
removed = all_words - clean_words
print(f"Stopwords removed: {removed}")

## Part 3: Feature Extraction — Bag of Words vs. TF-IDF

Now we turn words into numbers! We'll compare two approaches side by side.

In [ ]:
# =============================================
# BAG OF WORDS vs TF-IDF — Two ways to turn words into numbers
# =============================================

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

# Three movie reviews
reviews = [
    "The movie was great and the acting was great",
    "Terrible movie with terrible acting and bad plot",
    "A wonderful film with amazing performances"
]

print("=" * 60)
print("APPROACH 1: Bag of Words (just count words)")
print("=" * 60)

bow = CountVectorizer(stop_words='english')
X_bow = bow.fit_transform(reviews)
words = bow.get_feature_names_out()

print(f"Vocabulary: {list(words)}")
print()
for i, review in enumerate(reviews):
    counts = X_bow.toarray()[i]
    nonzero = [(w, c) for w, c in zip(words, counts) if c > 0]
    print(f'Review {i+1}: "{review[:50]}..."')
    print(f'  Counts: {dict(nonzero)}')
    print()

print("=" * 60)
print("APPROACH 2: TF-IDF (reward rare, important words)")
print("=" * 60)

tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(reviews)
words_tfidf = tfidf.get_feature_names_out()

for i, review in enumerate(reviews):
    scores = X_tfidf.toarray()[i]
    nonzero = [(w, f"{s:.3f}") for w, s in zip(words_tfidf, scores) if s > 0]
    nonzero.sort(key=lambda x: float(x[1]), reverse=True)
    print(f'Review {i+1}: "{review[:50]}..."')
    print(f'  TF-IDF scores: {dict(nonzero)}')
    print()

print("Notice: Words like 'wonderful' and 'amazing' (unique to one review)")
print("get HIGHER TF-IDF scores than 'movie' (appears in multiple reviews).")

## Part 4: Sentiment Analysis with scikit-learn

Now let's build a real sentiment classifier! We'll use the IMDB movie review dataset (50,000 reviews) with TF-IDF + Logistic Regression.

This uses the **exact same** `fit()` / `predict()` pattern you've been using all year.

In [ ]:
# =============================================
# SENTIMENT ANALYSIS WITH SCIKIT-LEARN
# =============================================
# TF-IDF + Logistic Regression — familiar tools!

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tensorflow import keras
import numpy as np
import time

# Step 1: Load IMDB dataset
print("Loading IMDB dataset...")
(x_train_idx, y_train), (x_test_idx, y_test) = keras.datasets.imdb.load_data(num_words=10000)

# Convert integer sequences back to text
word_index = keras.datasets.imdb.get_word_index()
reverse_index = {v + 3: k for k, v in word_index.items()}
reverse_index[0] = ''    # padding
reverse_index[1] = ''    # start token
reverse_index[2] = ''    # unknown

def decode_review(encoded):
    return ' '.join([reverse_index.get(i, '?') for i in encoded])

# Decode all reviews back to text
print("Decoding reviews to text...")
train_texts = [decode_review(r) for r in x_train_idx]
test_texts = [decode_review(r) for r in x_test_idx]

print(f"  Training reviews: {len(train_texts)}")
print(f"  Test reviews: {len(test_texts)}")
print(f"\nSample review (first 200 chars):")
print(f'  "{train_texts[0][:200]}..."')
print(f"  Label: {'Positive' if y_train[0] == 1 else 'Negative'}")

# Step 2: TF-IDF feature extraction
print("\nExtracting TF-IDF features...")
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)
print(f"  Feature matrix shape: {X_train.shape}")

# Step 3: Train Logistic Regression
print("\nTraining Logistic Regression...")
start = time.time()
model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train, y_train)
train_time = time.time() - start

# Step 4: Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n{'=' * 50}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Training Time: {train_time:.1f} seconds")
print(f"{'=' * 50}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# =============================================
# TEST WITH YOUR OWN REVIEWS!
# =============================================
# Write your own movie reviews and see what the model predicts

my_reviews = [
    "This movie was absolutely fantastic! Best film I have seen all year.",
    "Worst movie ever. Boring plot, terrible acting, complete waste of time.",
    "It was okay. Some parts were good but the ending was disappointing.",
    "The special effects were amazing but the story was weak and predictable.",
    "I laughed, I cried, I was on the edge of my seat the entire time!",
]

print("Testing your own reviews:\n")
X_new = tfidf.transform(my_reviews)
predictions = model.predict(X_new)
probabilities = model.predict_proba(X_new)

for review, pred, probs in zip(my_reviews, predictions, probabilities):
    sentiment = "POSITIVE" if pred == 1 else "NEGATIVE"
    confidence = max(probs) * 100
    print(f'  "{review}"')
    print(f'  --> {sentiment} ({confidence:.1f}% confident)')
    print()

In [ ]:
# =============================================
# TOP WORDS FOR POSITIVE AND NEGATIVE REVIEWS
# =============================================
# Which words does the model associate with each sentiment?

import numpy as np

feature_names = np.array(tfidf.get_feature_names_out())
coefficients = model.coef_[0]

# Top 15 positive words (highest coefficients)
top_positive_idx = np.argsort(coefficients)[-15:][::-1]
print("Top 15 words associated with POSITIVE reviews:")
for idx in top_positive_idx:
    print(f"  {feature_names[idx]:>15}  (score: {coefficients[idx]:.3f})")

print()

# Top 15 negative words (lowest coefficients)
top_negative_idx = np.argsort(coefficients)[:15]
print("Top 15 words associated with NEGATIVE reviews:")
for idx in top_negative_idx:
    print(f"  {feature_names[idx]:>15}  (score: {coefficients[idx]:.3f})")

## Part 5: Sentiment Analysis with Keras

Now let's try the neural network approach! Instead of TF-IDF, Keras will use **word embeddings** — it learns its own number representation for each word during training.

In [ ]:
# =============================================
# SENTIMENT ANALYSIS WITH KERAS
# =============================================
# Neural network with word embeddings!

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import time

# Step 1: Load IMDB dataset
print("Loading IMDB dataset...")
max_words = 10000     # Top 10,000 most common words
max_len = 200         # Pad/truncate reviews to 200 words

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_words)
print(f"  Training reviews: {len(x_train)}")
print(f"  Test reviews: {len(x_test)}")

# Step 2: Pad sequences to equal length
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len)
print(f"  Each review padded to {max_len} tokens")

# Step 3: Build the model
model = keras.Sequential([
    layers.Embedding(max_words, 32, input_length=max_len),  # Learn word vectors
    layers.GlobalAveragePooling1D(),                         # Average all word vectors
    layers.Dense(64, activation="relu"),                     # Hidden layer
    layers.Dense(1, activation="sigmoid")                    # Output: positive or negative
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
print("\nModel architecture:")
model.summary()

# Step 4: Train
print("\nTraining...")
start = time.time()
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
train_time = time.time() - start

# Step 5: Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n{'=' * 50}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Training Time: {train_time:.1f} seconds")
print(f"{'=' * 50}")

In [ ]:
# =============================================
# VISUALIZE TRAINING PROGRESS
# =============================================

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax1.plot(history.history['accuracy'], label='Training', marker='o')
ax1.plot(history.history['val_accuracy'], label='Validation', marker='s')
ax1.set_title('Model Accuracy Over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Training', marker='o')
ax2.plot(history.history['val_loss'], label='Validation', marker='s')
ax2.set_title('Model Loss Over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Side-by-Side Comparison

Let's compare both approaches on the same dataset!

In [ ]:
# =============================================
# FINAL COMPARISON: scikit-learn vs Keras
# =============================================

print("=" * 60)
print("FINAL SCOREBOARD: NLP Sentiment Analysis")
print("=" * 60)
print()
print(f"{'Approach':<30} {'Accuracy':<12} {'Type'}")
print("-" * 60)
print(f"{'TF-IDF + Logistic Regression':<30} {accuracy * 100:.2f}%       Classic ML")
print(f"{'Keras (Embeddings + NN)':<30} {test_acc * 100:.2f}%       Neural Network")
print("-" * 60)
print()
print("Key Differences:")
print("  - scikit-learn uses TF-IDF (we engineer the features)")
print("  - Keras uses Embeddings (the network learns its own features)")
print("  - Both achieve similar accuracy on this task!")
print("  - Neural networks shine more on complex tasks like translation")
print()
print("Coming next: Transformers — the architecture behind ChatGPT and Claude!")